# Introduction
Urban Rivers is a conservation organization helping to restore the Chicago River.  

Part of the project involves tracking changes in biodiversity attributable to the installation of floating wetlands.  
Volunteers have placed and maintain motion detection cameras (camera traps) along installations, natural river banks, and the existing metal retaining walls on the water way.

These pictures are available on s3 - this code investigates downloading a portion of those images for testing with SpeciesNet.

> Removed geofencing, using hashmd5 for image names, full dataset, clean when finished.  
> Connects to production mongo.  
> Updates and persists json and csv files for continual processing into master json.  
> 2025-07-06 Testing with min 1280px width in 10k batches


This workbook was used for detections using Kaggle's GPUs and is linked therein.  
https://www.kaggle.com/code/morescope/speciesnet-testing-urbanrivers

## Notebook Setup and Required Packages

In [1]:
# Data Handling
import pandas as pd
import numpy as np

# IO - getting files and images from MongoDB and S3
from pymongo import MongoClient
from kaggle_secrets import UserSecretsClient
import requests

from concurrent.futures import ThreadPoolExecutor, as_completed

from pathlib import Path
from PIL import Image
from io import BytesIO

import os
import sys
import re
import shutil
import json
import time
from datetime import datetime

# Move speciesnet install to where it's used

from IPython.display import display, HTML, JSON, Javascript

import kagglehub

print("Libraries Loaded")

Libraries Loaded


In [2]:
# Configuration for Multithreading and Batching
num_batches = 10
max_threads = 20

# Prepare folders
output_root = Path("output")
output_root.mkdir(exist_ok=True)
images_root = Path("images")
images_root.mkdir(exist_ok=True)

In [3]:
# Add persistent data if available
!cp /kaggle/input/ur-speciesnet-persistent/output/predictions_dict_master.json /kaggle/working/output/predictions_dict_master.json

## Access The URIs from S3 through MongoDB

In [4]:
# Get the stored mongo uri secret
user_secrets = UserSecretsClient()
mongo_uri = user_secrets.get_secret("MONGO_PROD")

# Connect to the MongoDB client
client = MongoClient(mongo_uri)
 
# Access the database and collection
db = client['test']
collection = db['cameratrapmedias'] 
 
# Query the collection to retrieve records with image URLs, metadata, and the first index of 'relativePath'
data = list(collection.aggregate([
    # {
    #     '$match': {
    #         'aiResults': None  # Match documents where aiResults is null or missing
    #     }
    # },
    {
        '$project': {
            '_id': 0,
            'publicURL': 1,
            'timestamp': 1,
            'folderName': { '$arrayElemAt': ['$relativePath', 1] },
            'fileName': 1,
            'mediaID': 1
        }
    },
    # { '$limit': 150 }
]))

# Stop execution if no records are found
if not data:
    display(HTML("<h3 style='color:red;'>No records found. Exiting notebook.</h3>"))
    sys.exit()
    !exit # Redundant full exit
 
# Convert the data to a pandas DataFrame for exploration
df = pd.DataFrame(data)

# preview df
display(df.head())
print(f'Rows: {len(df)}')

,mediaID,timestamp,publicURL,fileName,folderName
0,c112813a5f3b9cec26f95fad982b8d09,2024-01-24 18:56:50,https://urbanriverrangers.s3.amazonaws.com/ima...,SYFW0001.JPG,2024-01-30_prologis_02
1,0647380f2d59692f5b2b642312844e9f,2024-01-24 19:01:54,https://urbanriverrangers.s3.amazonaws.com/ima...,SYFW0002.JPG,2024-01-30_prologis_02
2,0db73c6c1efb4968c04a47e418ebeefb,2024-01-24 19:03:05,https://urbanriverrangers.s3.amazonaws.com/ima...,SYFW0004.JPG,2024-01-30_prologis_02
3,31fc53de29056b4dd8bc7b1804617f00,2024-01-24 19:04:19,https://urbanriverrangers.s3.amazonaws.com/ima...,SYFW0006.JPG,2024-01-30_prologis_02
4,14664d764836c5fd9a38284dc6103527,2024-01-24 19:05:33,https://urbanriverrangers.s3.amazonaws.com/ima...,SYFW0008.JPG,2024-01-30_prologis_02


Rows: 229286


In [5]:
# Export the production array to a CSV file 
df.to_csv('ur_test_medias_prod.csv', index=False)

In [6]:
# load the existing predictions_dict to filter out media IDs already predicted
predictions_file = Path("output/predictions_dict_master.json")
with open(predictions_file, "r") as f:
    predictions_data = json.load(f)

# Extract mediaIDs from filepaths in predictions
predicted_filepaths = [p["filepath"] for p in predictions_data.get("predictions", [])]
predicted_media_ids = {Path(fp).stem for fp in predicted_filepaths}

print(f"Found {len(predicted_media_ids)} predicted mediaIDs")

# Filter out rows where mediaID is already in the predictions
initial_count = len(df)
df_filtered = df[~df["mediaID"].astype(str).isin(predicted_media_ids)]
filtered_count = len(df_filtered)

print(f"Filtered out {initial_count - filtered_count} rows. Remaining: {filtered_count}")

Found 60010 predicted mediaIDs
Filtered out 60010 rows. Remaining: 169276


In [7]:
# Save the filtered df
df_filtered.to_csv("ur_test_medias_prod_filtered.csv", index=False)

# Filtered out files to download and predict
The persistent files for predictions.json that continue to grow will serve as a means of not redownloading images.
Now that we have a connection to the MongoDB server and access to the URLs, let's use the download images.

# Max Images at 1280px resolution
Trial and error puts this at some point after 70k - so we'll back off to 50k to be conservative

In [8]:
# Each time running - just process the next 10k files
if filtered_count > 50000:
    df_to_run = df_filtered[:50000]
else:
    df_to_run = df_filtered

## Download Images

In [9]:
%%time
# Create a directory to save the images - redundant but ok if in testing
output_root.mkdir(exist_ok=True)
path = Path('images')
path.mkdir(exist_ok=True)

# Optional - define chunks - for each run, the first n rows will be processed
df_download = df_to_run # up to 50k images based on total amount remaining
print(f'Peparing to Download {len(df_download)} images')

# Create a tool for resizing so cropping top and bottom can happen while keeping the aspect ratio
def resize_to_height(image, target_width=1280):
    og_width, og_height = image.size
    new_height = int(og_height * (target_width / og_width))
    return image.resize((target_width, new_height))

# Tool for download and processing
def process_row(row, dest_folder, session):
    url = row['publicURL']
    filename = f"{row['mediaID']}.jpg"
    dest = dest_folder / filename

    try:
        response = session.get(url, timeout=5)
        response.raise_for_status()

        image = Image.open(BytesIO(response.content)).convert("RGB")
        image = resize_to_height(image, target_width=1280)
        image.save(dest, format="JPEG", quality=75)
    except Exception as e:
        print(f"failed to process {filename}: {e}")

for batch_idx, df_chunk in enumerate(np.array_split(df_download, num_batches)):
    batch_folder = images_root / f'batch_{batch_idx}'
    batch_folder.mkdir(exist_ok=True)
    print(f'Processing batch {batch_idx + 1} / {num_batches}...')

    rows = df_chunk.to_dict(orient='records')
    start = time.time()

    with requests.Session() as session:
        with ThreadPoolExecutor(max_workers=max_threads) as executor:
            futures = [executor.submit(process_row, row, batch_folder, session) for row in rows]
            for future in as_completed(futures):
                future.result()  # you can add error catching here if needed

    print(f"Batch {batch_idx+1} took {time.time() - start:.2f} seconds.")
        
print(f'{len(df_download)} Images Downloaded and Resized')

Peparing to Download 50000 images
Processing batch 1 / 10...


/usr/local/lib/python3.11/dist-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Batch 1 took 692.40 seconds.
Processing batch 2 / 10...
Batch 2 took 510.31 seconds.
Processing batch 3 / 10...
Batch 3 took 140.68 seconds.
Processing batch 4 / 10...
Batch 4 took 484.39 seconds.
Processing batch 5 / 10...
Batch 5 took 786.32 seconds.
Processing batch 6 / 10...
Batch 6 took 301.63 seconds.
Processing batch 7 / 10...
Batch 7 took 152.44 seconds.
Processing batch 8 / 10...
Batch 8 took 185.82 seconds.
Processing batch 9 / 10...
Batch 9 took 329.79 seconds.
Processing batch 10 / 10...
Batch 10 took 684.08 seconds.
50000 Images Downloaded and Resized
CPU times: user 4h 1min 34s, sys: 29min 31s, total: 4h 31min 5s
Wall time: 1h 11min 8s


In [10]:
# Uncomment and run this if the images need to be redone
# !rm images -r
# !rm output/docs -r
# !rm docs.zip
# %lsmagic

## Running Species Net on the Full Dataset
Now that we have the max number of images downloaded (19.5GB) let's run speciesnet

Note there might be a better way of doing this using bytes downloaded from s3 - but I haven't figured that part out yet.

### We're going to try a multithreading chunks approach

In [11]:
# Install speciesnet and related megadetector libraries
!pip install -Uqq speciesnet megadetector-utils
from speciesnet import SpeciesNet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 70.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 7.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.7/93.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 652.2/652.2 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 102.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [12]:
def print_predictions(predictions_dict: dict) -> None:
    print("Predictions:")
    for prediction in predictions_dict["predictions"][0:1]:
        print(prediction["filepath"], "=>", prediction["prediction"])

### Download Model

In [13]:
# Choose the folder we're going to download the model to
model_path = '/content/models'
os.makedirs(model_path, exist_ok=True)

# Download the model (it will go to a folder like /kaggle/input/...)
download_path = kagglehub.model_download('google/speciesnet/PyTorch/v4.0.1a',
                                          force_download=True)

print('Model downloaded to temporary folder: {}'.format(download_path))

# List the contents of the downloaded directory to identify the actual files/subdirectories
model_files = os.listdir(download_path)

# Copy the contents of the model file to our destination folder
for item_name in model_files:
    source_path = os.path.join(download_path, item_name)
    destination_path = os.path.join(model_path, item_name)
    if os.path.isfile(source_path):
        shutil.copy2(source_path, destination_path)
    elif os.path.isdir(source_path):
        shutil.copytree(source_path, destination_path, dirs_exist_ok=True)

print('{} files copied to: {}'.format(len(model_files),model_path))

Model downloaded to temporary folder: /kaggle/input/speciesnet/pytorch/v4.0.1a/1
6 files copied to: /content/models


In [14]:
# Pick the model we want to use (4.0.1a)
model = SpeciesNet(model_path)

print('Model Loaded')

Model Loaded


In [15]:
# Let's format a request string as a list of dicts (aka JSON string format)
def create_instances(batch_folder):
    image_paths = [f'{batch_folder}/{f}' for f in os.listdir(batch_folder) if f.lower().endswith('.jpg')]

    instances = []
    for image_path in image_paths:
        instances.append({
            'filepath': image_path
        })

    # Check that it's saved correctly by verifying the first
    print(instances[0:1])

    return instances


for batch_index in range(len(os.listdir(images_root))):
    instances = create_instances(f'{images_root}/batch_{batch_index}')

    # make the predictions and get a sense of how long it would take
    %time predictions_dict = model.predict(instances_dict={"instances": instances})

    print_predictions(predictions_dict) # show the first prediction of each batch

    # Save the dict to the batch folder
    with open(f'{images_root}/batch_{batch_index}/predictions_dict_{batch_index}.json', 'w') as f:
        json.dump(predictions_dict, f, indent=2)

    print(f'predictions_dict_{batch_index}.json saved to {images_root}/batch_{batch_index}')

[{'filepath': 'images/batch_0/2638570bb09c02772b07e6a7cf8360f7.jpg'}]
CPU times: user 20min 12s, sys: 29.7 s, total: 20min 41s
Wall time: 9min 32s
Predictions:
images/batch_0/2638570bb09c02772b07e6a7cf8360f7.jpg => 446887df-3477-4f4a-a434-852f96ba48d9;reptilia;testudines;emydidae;emys;marmorata;western pond turtle
predictions_dict_0.json saved to images/batch_0
[{'filepath': 'images/batch_1/8662ac0a13e3b135d3ff35949c12af48.jpg'}]
CPU times: user 18min 47s, sys: 52.6 s, total: 19min 40s
Wall time: 9min 21s
Predictions:
images/batch_1/8662ac0a13e3b135d3ff35949c12af48.jpg => b1352069-a39c-4a84-a949-60044271c0c1;aves;;;;;bird
predictions_dict_1.json saved to images/batch_1
[{'filepath': 'images/batch_2/cc01be2cbe3398a773292241eeaf096c.jpg'}]
CPU times: user 17min 44s, sys: 40.6 s, total: 18min 24s
Wall time: 8min 24s
Predictions:
images/batch_2/cc01be2cbe3398a773292241eeaf096c.jpg => f1856211-cfb7-4a5b-9158-c0f72fd09ee6;;;;;;blank
predictions_dict_2.json saved to images/batch_2
[{'filepath

## Let's save the predictions dict json file

In [16]:
%%time
# To concatenate all the json files
output_file = output_root / "predictions_dict_master.json" # Uncomment when loading from start

# Initialize the master predictions list
master_predictions = []

# Load existing master file if it exists
if output_file.exists():
    with open(output_file, "r") as f:
        existing_data = json.load(f)
        if "predictions" in existing_data:
            master_predictions.extend(existing_data["predictions"])
        else:
            print(f"{output_file} missing 'predictions' key")

# Use today's date
rundate = datetime.now().date().isoformat()

# Loop through files matching the pattern
for json_file in sorted(images_root.glob("batch_*/predictions_dict_*.json")):
    with open(json_file, "r") as f:
        data = json.load(f)
        if "predictions" in data:
            for record in data["predictions"]:
                record["run_date"] = rundate # adding a run_date field to each predictions record
                master_predictions.append(record)  # Concatenate predictions!
        else:
            print(f"{json_file} missing 'predictions' key")

# Deduplicate based on media ID extracted from 'filepath'
deduped_predictions = {
    Path(record["filepath"]).stem: record
    for record in master_predictions
}
master_predictions = list(deduped_predictions.values())

# Write the combined predictions to a new file
with open(output_file, "w") as f:
    json.dump({"predictions": master_predictions}, f, indent=2)

print(f"Combined {len(master_predictions)} predictions into {output_file}")


Combined 110010 predictions into output/predictions_dict_master.json
CPU times: user 10 s, sys: 686 ms, total: 10.7 s
Wall time: 11.1 s


### Final Cleanup of Files
Remove all images because we are adding files persistence

In [17]:
# Remove the image directories because nobody needs to store them at the end here
shutil.rmtree('/kaggle/working/images')

print("Files cleaned up")

Files cleaned up
